# 玉山證券行情 API — 台股盤前試撮資料擷取 (Web API 快照)

本筆記本示範如何使用**玉山證券行情 Web API（REST API）**取得台股**盤前試撮（08:30 ~ 09:00）**的即時快照數據。

### 為什麼使用 Web API 而非 WebSocket？
- **Web API (HTTP REST)**：適合在「特定時間點（例如 08:50 或 08:58）」進行單次快照查詢，取得當下的試撮價、試撮量、最佳五檔委買委賣及試撮明細，不需維護長時間 WebSocket 雙向連線與心跳。
- **配額與限制**：玉山 Web API 日內行情與快照端點限制為 **600 次 / 分鐘**，單次查詢極為快速且配額充裕。

---

### 盤前試撮（Trial Match）三大核心端點：
1. **`rest_stock.intraday.quote(symbol)`**：取得個股最新即時報價，內含：
   - `isTrial`: 當前是否為試撮階段（08:30~09:00 或 13:25~13:30 為 `True`）。
   - `lastTrial`: 最後一筆試撮成交資訊（含 `price`, `size`, `bid`, `ask`, `time`, `serial`）。
   - `bids` / `asks`: 試撮當下的最佳五檔委託買賣價量。
   - `lastPrice`, `lastSize`, `change`, `changePercent`: 最新試撮價與漲跌幅。
2. **`rest_stock.intraday.trades(symbol, isTrial=True)`**：專門查詢**試撮逐筆成交明細**，可取得 08:30 以來所有試撮 tick 紀錄（預設降冪排序，第 0 筆即為最新一筆試撮）。
3. **`rest_stock.snapshot.quotes(market="TSE")`**：全市場行情快照，在試撮期間可一次取得全市場（或特定清單）所有標的的最新試撮成交價量，避免重複呼叫單檔 API。

## 1. 環境準備與憑證設定說明

### (1) 安裝玉山行情 SDK (`esun_marketdata`)
玉山提供專屬的 Python wheel 安裝檔（已存放於 `../esun api說明/` 資料夾）：
```bash
pip install "../esun api說明/esun_marketdata-2.2.0-cp37-abi3-win_amd64.whl"
```

### (2) 設定檔 (`config.ini`) 格式
玉山行情 API 採用 `config.ini` 管理連線端點、憑證與 API 金鑰。
格式範例如下：
```ini
[Core]
Entry = https://esuntradingapi-simulation.esunsec.com.tw/api/v1
Environment = SIMULATION

[Cert]
Path = ../esun api說明/cert.p12

[Api]
Key = 你的_API_KEY
Secret = 你的_API_SECRET

[User]
Account = 你的身分證字號或玉山綜合帳號
```

> **提示**：
> - 若您參與的是競賽模擬環境，`[Core]` 的 `Environment` 請設為 `SIMULATION`。
> - `[Cert]` 的 `Path` 請填入您的 `cert.p12` 憑證路徑（支援相對路徑或絕對路徑）。
> - 若在 `../esun api說明/` 下已有 `config.simulation.ini` 與 `cert.p12`，可直接指定該檔案路徑。

In [2]:
# %pip install "../esun api說明/esun_marketdata-2.2.0-cp37-abi3-win_amd64.whl"

In [1]:
import os
from pathlib import Path
from configparser import ConfigParser
import pandas as pd
from datetime import datetime

# 設定 config.ini 路徑
# 預設先檢查本資料夾的 config.ini，若不存在則檢查 esun api說明 資料夾下的 config.simulation.ini
current_dir = Path.cwd()
candidate_configs = [
    current_dir / "config.ini",
    current_dir.parent / "esun api說明" / "config.simulation.ini",
    current_dir.parent / "esun api說明" / "config.ini",
]

config_path = None
for p in candidate_configs:
    if p.exists():
        config_path = p
        break

if config_path:
    print(f"[OK] 找到設定檔: {config_path}")
else:
    config_path = current_dir / "config.ini"
    print(f"[提示] 尚未建立 config.ini，請依下方說明建立: {config_path}")

[OK] 找到設定檔: h:\我的雲端硬碟\DataScience\2026玉山人工智慧賽\esun api說明\config.simulation.ini


## 2. 初始化玉山行情 SDK 並登入

In [2]:
from esun_marketdata import EsunMarketdata

# 讀取設定檔
config = ConfigParser()
config.read(str(config_path), encoding="utf-8")

# 檢查憑證路徑是否存在（若設定檔中使用相對路徑，協助解析為相對於設定檔所在目錄）
cert_rel_path = config.get("Cert", "Path", fallback="")
cert_file = Path(cert_rel_path)
if not cert_file.is_absolute():
    cert_file = (config_path.parent / cert_rel_path).resolve()

print(f"檢查憑證路徑: {cert_file} (存在: {cert_file.exists()})")

# 更新設定檔中的憑證絕對路徑，確保 SDK 能夠正確載入
config.set("Cert", "Path", str(cert_file))

# 初始化 SDK 並執行登入驗證
sdk = EsunMarketdata(config)
sdk.login()
print("[OK] 玉山證券行情 SDK 登入成功！")

# 取得 Web API 股票操作客戶端 (REST Client)
rest_stock = sdk.rest_client.stock

檢查憑證路徑: H:\我的雲端硬碟\DataScience\2026玉山人工智慧賽\esun api說明\cert.p12 (存在: True)
[OK] 玉山證券行情 SDK 登入成功！


## 3. 個股即時試撮快照 (Quote)

透過 `rest_stock.intraday.quote(symbol="2330")` 可取得：
- `isTrial`: 是否正處於試撮階段
- `lastTrial`: 最後一筆試撮成交價量與撮合時間
- `bids` / `asks`: 試撮最佳五檔委買委賣價量
- `lastPrice`, `lastSize`, `change`, `changePercent`: 當前撮合統計

In [ ]:
def format_micro_timestamp(us_ts):
    """將微秒時間戳轉換為標準日期時間字串 (YYYY-MM-DD HH:MM:SS.ffffff)"""
    if not us_ts:
        return "-"
    return datetime.fromtimestamp(us_ts / 1_000_000).strftime("%Y-%m-%d %H:%M:%S.%f")

def get_trial_quote(symbol: str) -> dict:
    """
    查詢指定股票代碼當下的即時試撮報價快照
    """
    res = rest_stock.intraday.quote(symbol=symbol)
    
    # 解析試撮基本資訊
    is_trial = res.get("isTrial", False)
    last_trial = res.get("lastTrial", {})
    
    summary = {
        "代碼": res.get("symbol"),
        "名稱": res.get("name"),
        "日期": res.get("date"),
        "是否試撮中 (isTrial)": is_trial,
        "最新試撮價": last_trial.get("price", res.get("lastPrice")),
        "最新試撮量(張)": last_trial.get("size", res.get("lastSize")),
        "試撮時間": format_micro_timestamp(last_trial.get("time")),
        "昨日收盤價": res.get("previousClose"),
        "試撮漲跌": res.get("change"),
        "試撮漲跌幅(%)": res.get("changePercent"),
        "是否漲停": res.get("isLimitUpPrice", False),
        "是否跌停": res.get("isLimitDownPrice", False),
        "最後更新時間": format_micro_timestamp(res.get("lastUpdated")),
    }
    
    # 解析最佳五檔試撮委託簿
    bids = res.get("bids", [])
    asks = res.get("asks", [])
    orderbook_rows = []
    for i in range(max(len(bids), len(asks), 5)):
        b_p = bids[i]["price"] if i < len(bids) else None
        b_s = bids[i]["size"] if i < len(bids) else None
        a_p = asks[i]["price"] if i < len(asks) else None
        a_s = asks[i]["size"] if i < len(asks) else None
        orderbook_rows.append({
            "檔位": f"第 {i+1} 檔",
            "委買量(張)": b_s,
            "委買價": b_p,
            "委賣價": a_p,
            "委賣量(張)": a_s,
        })
    
    df_orderbook = pd.DataFrame(orderbook_rows)
    
    return {
        "raw": res,
        "summary": summary,
        "orderbook": df_orderbook,
    }

# 測試取得台積電 (2330) 試撮報價
quote_data = get_trial_quote("2330")

print("=== 試撮報價摘要 ===")
for k, v in quote_data["summary"].items():
    print(f"{k}: {v}")

print("\n=== 試撮最佳五檔 (Orderbook) ===")
quote_data["orderbook"]

=== 試撮報價摘要 ===
代碼: 4938
名稱: 和碩
日期: 2026-11-03
是否試撮中 (isTrial): False
最新試撮價: 101.5
最新試撮量(張): 1337
試撮時間: 2026-11-03 13:29:59.549491
昨日收盤價: 102.5
試撮漲跌: -1
試撮漲跌幅(%): -0.98
是否漲停: False
是否跌停: False
最後更新時間: 2026-11-03 13:30:00.000000

=== 試撮最佳五檔 (Orderbook) ===


      檔位  委買量(張)    委買價    委賣價  委賣量(張)
0  第 1 檔      79  101.5  102.0     946
1  第 2 檔     283  101.0  102.5    1344
2  第 3 檔     763  100.5  103.0     529
3  第 4 檔     319  100.0  103.5     385
4  第 5 檔      60   99.9  104.0     598

## 4. 個股盤前試撮逐筆明細 (Trades, isTrial=True)

在 08:30 ~ 09:00 期間，交易所會定期（約每數秒）產生一次試撮撮合結果。
透過帶入 `isTrial=True` 參數：
```python
rest_stock.intraday.trades(symbol="2330", isTrial=True, limit=20)
```
可以取得當天截至目前為止的**試撮成交紀錄歷史序列**，最上方（第 0 筆）即為發送請求時最新一次試撮。

In [6]:
def get_trial_trades(symbol: str, limit: int = 20) -> pd.DataFrame:
    """
    查詢個股盤前/盤後試撮逐筆成交明細
    參數:
        symbol: 股票代碼
        limit: 取得筆數限制（預設 20 筆）
    回傳:
        pd.DataFrame (按時間降冪排序，最上面為最新試撮)
    """
    res = rest_stock.intraday.trades(symbol=symbol, isTrial=True, limit=limit, sort="desc")
    data = res.get("data", [])
    
    if not data:
        print(f"[提示] 目前標的 {symbol} 尚無試撮逐筆明細（可能非試撮時段 08:30~09:00 或市場尚未產生試撮）")
        return pd.DataFrame()
    
    records = []
    for tick in data:
        records.append({
            "代碼": symbol,
            "時間": format_micro_timestamp(tick.get("time")),
            "試撮成交價": tick.get("price"),
            "試撮成交單量(張)": tick.get("size"),
            "累計試撮量(張)": tick.get("volume"),
            "成交買價": tick.get("bid"),
            "成交賣價": tick.get("ask"),
            "流水號": tick.get("serial"),
        })
    
    df_trades = pd.DataFrame(records)
    return df_trades

# 測試取得台積電 (2330) 最近 10 筆試撮紀錄
df_trial_ticks = get_trial_trades("2330", limit=10)
df_trial_ticks.head()

     代碼                          時間  試撮成交價  試撮成交單量(張) 累計試撮量(張)  成交買價  成交賣價  \
0  2330  2026-10-22 13:29:59.836832   2385       4826     None  2385  2390   
1  2330  2026-10-22 13:29:54.822253   2385       4671     None  2385  2390   
2  2330  2026-10-22 13:29:49.806637   2385       4663     None  2385  2390   
3  2330  2026-10-22 13:29:44.791056   2385       4672     None  2385  2390   
4  2330  2026-10-22 13:29:39.774422   2385       4693     None  2385  2390   

        流水號  
0  18918410  
1  18917628  
2  18916966  
3  18916332  
4  18915726  

## 5. 多檔標的批次試撮快照 (Batch Trial Snapshot)

在實際策略或競賽中，我們通常需要關注多檔標的（例如玉山挑戰賽規定的 150 檔標的或投資組合持股）。

有兩種方式可以取得多檔標的試撮：
1. **方式 A：逐檔查詢 `quote(symbol)`**（靈活，且能取得完整五檔與 `lastTrial` 細節，需留意 600 次/分頻率）。
2. **方式 B：全市場快照 `snapshot.quotes(market="TSE")`**（單一請求直接取得全市場快照，再過濾所需標的，最省 API 配額）。

In [11]:
def get_batch_trial_quotes(symbols: list) -> pd.DataFrame:
    """
    方式 A：批次查詢指定股票清單之試撮概況
    """
    rows = []
    for sym in symbols:
        try:
            res = rest_stock.intraday.quote(symbol=sym)
            last_trial = res.get("lastTrial", {})
            rows.append({
                "代碼": res.get("symbol"),
                "名稱": res.get("name"),
                "試撮價": last_trial.get("price", res.get("lastPrice")),
                "試撮量(張)": last_trial.get("size", res.get("lastSize")),
                "昨收價": res.get("previousClose"),
                "漲跌": res.get("change"),
                "漲跌幅(%)": res.get("changePercent"),
                "試撮中": res.get("isTrial", False),
                "試撮時間": format_micro_timestamp(last_trial.get("time")),
            })
        except Exception as e:
            rows.append({
                "代碼": sym,
                "名稱": "查詢失敗",
                "錯誤原因": str(e)
            })
    return pd.DataFrame(rows)

def get_market_snapshot_trial(market: str = "TSE", target_symbols: list = None) -> pd.DataFrame:
    """
    方式 B：呼叫全市場快照 API，並篩選目標股票（單次請求即可取得全市場行情）
    market: 可選 'TSE'（上市）或 'OTC'（上櫃）
    """
    res = rest_stock.snapshot.quotes(market=market)
    data = res.get("data", [])
    
    target_set = set(target_symbols) if target_symbols else None
    rows = []
    for item in data:
        sym = item.get("symbol")
        if target_set and sym not in target_set:
            continue
        rows.append({
            "代碼": sym,
            "名稱": item.get("name"),
            "試撮成交價": item.get("closePrice"),
            "漲跌": item.get("change"),
            "漲跌幅(%)": item.get("changePercent"),
            "成交量(張)": item.get("tradeVolume"),
            "成交金額": item.get("tradeValue"),
            "更新時間": format_micro_timestamp(item.get("lastUpdated")),
        })
    return pd.DataFrame(rows)

# 範例清單：大型權值股與熱門 ETF
watchlist = ["2330", "2317", "2454", "0050", "0056"]

print("=== 批次個別試撮快照 ===")
df_batch = get_batch_trial_quotes(watchlist)
df_batch

=== 批次個別試撮快照 ===


     代碼      名稱     試撮價  試撮量(張)      昨收價      漲跌  漲跌幅(%)    試撮中  \
0  2330     台積電  2385.0    4826  2420.00  -35.00   -1.45  False   
1  2317      鴻海   245.0    3111   250.00   -5.00   -2.00  False   
2  2454     聯發科  4010.0     758  4140.00 -130.00   -3.14  False   
3  0050  元大台灣50   106.9    2404   108.15   -1.25   -1.16  False   
4  0056   元大高股息    56.3    1060    57.05   -0.75   -1.31  False   

                         試撮時間  
0  2026-10-22 13:29:59.836832  
1  2026-10-22 13:29:59.836832  
2  2026-10-22 13:29:59.636141  
3  2026-10-22 13:29:59.985679  
4  2026-10-22 13:29:59.383569  

## 6. 整合儀表板函式：一鍵檢視持股盤前試撮狀態

在開盤前（08:30 ~ 09:00），您可以直接呼叫以下封裝函式 `check_premarket_dashboard`，一次檢視所有關注標的的試撮成交價、買賣一檔報價與委託價差。

In [8]:
def check_premarket_dashboard(symbols: list) -> pd.DataFrame:
    """
    盤前試撮整合儀表板
    整理：代碼、名稱、昨收、試撮價、漲跌幅、委買一(價/量)、委賣一(價/量)、價差、試撮狀態
    """
    rows = []
    for sym in symbols:
        try:
            q = rest_stock.intraday.quote(symbol=sym)
            last_trial = q.get("lastTrial", {})
            bids = q.get("bids", [])
            asks = q.get("asks", [])
            
            bid1_price = bids[0]["price"] if bids else None
            bid1_size = bids[0]["size"] if bids else None
            ask1_price = asks[0]["price"] if asks else None
            ask1_size = asks[0]["size"] if asks else None
            
            spread = round(ask1_price - bid1_price, 2) if (ask1_price and bid1_price) else None
            
            rows.append({
                "代碼": sym,
                "名稱": q.get("name"),
                "昨收價": q.get("previousClose"),
                "最新試撮價": last_trial.get("price", q.get("lastPrice")),
                "試撮成交量": last_trial.get("size", q.get("lastSize")),
                "漲跌幅(%)": q.get("changePercent"),
                "買一價": bid1_price,
                "買一量": bid1_size,
                "賣一價": ask1_price,
                "賣一量": ask1_size,
                "買賣價差": spread,
                "試撮中": "是" if q.get("isTrial") else "否",
                "時間": format_micro_timestamp(last_trial.get("time") or q.get("lastUpdated")),
            })
        except Exception as err:
            rows.append({"代碼": sym, "錯誤": str(err)})
            
    df_dashboard = pd.DataFrame(rows)
    return df_dashboard

# 執行盤前儀表板檢視
df_result = check_premarket_dashboard(["2330", "2317", "2454", "0050"])
df_result

     代碼      名稱      昨收價   最新試撮價  試撮成交量  漲跌幅(%)     買一價   買一量      賣一價  賣一量  \
0  2330     台積電  2420.00  2385.0   4826   -1.45  2385.0   679  2390.00  564   
1  2317      鴻海   250.00   245.0   3111   -2.00   245.0    29   245.50  303   
2  2454     聯發科  4140.00  4010.0    758   -3.14  4010.0    13  4020.00    1   
3  0050  元大台灣50   108.15   106.9   2404   -1.16   106.9  1063   106.95  559   

    買賣價差 試撮中                          時間  
0   5.00   否  2026-10-22 13:29:59.836832  
1   0.50   否  2026-10-22 13:29:59.836832  
2  10.00   否  2026-10-22 13:29:59.636141  
3   0.05   否  2026-10-22 13:29:59.985679  

## 7. 盤前試撮重點機制與注意事項

1. **時段規則**：
   - **08:30 ~ 09:00**：台股開盤前試撮時段。自 08:30 開始接受買賣申報，約每 5 秒進行一次模擬撮合並揭示試撮行情與五檔。
   - **08:59 ~ 09:00**：開盤前一分鐘不得撤銷或變更買賣申報（暫緩開盤機制適用標的除外）。
   - **09:00**：正式開盤並進行集合競價第一筆撮合，隨後進入逐筆交易模式。
2. **資料判斷依據**：
   - `isTrial == True`：代表回傳之報價為模擬撮合狀態。
   - 若在非試撮時段呼叫 `quote(symbol)`，`isTrial` 會為 `False`，此時 `lastPrice` 為盤中逐筆成交價或昨日收盤價；而 `lastTrial` 仍會保留當天最後一次試撮的撮合紀錄。
3. **API 頻率管制**：
   - REST Web API 股票日內行情上限為 **600 次 / 分鐘**。若監控大量標的，建議採用 `get_market_snapshot_trial`（全市場快照一次抓回）或適度加入間隔，避免觸發 HTTP 429。
4. **憑證資安守則**：
   - 請切勿將含有身分證字號、API Secret 或 `cert.p12` 的檔案上傳至 GitHub 或公開儲存庫（`.gitignore` 已設定自動忽略 `*.ini` 與 `*.p12`）。